In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [2]:
import requests

print(requests.get("https://www.google.com").status_code)

200


In [3]:
!git clone https://github.com/pjjoseph10/EcoDysformer

fatal: destination path 'EcoDysformer' already exists and is not an empty directory.


In [4]:
%cd EcoDysformer
!pip install -r requirements.txt

/kaggle/working/EcoDysformer


In [5]:
from pathlib import Path

def patch(relpath, replaces):
    p = Path(relpath); t = p.read_text(encoding="utf-8"); changed = False
    for old, new in replaces:
        if new in t:                       # already patched -> skip (idempotent)
            continue
        assert old in t, f"ANCHOR NOT FOUND in {relpath}:\n{old[:70]}..."
        t = t.replace(old, new, 1); changed = True
    p.write_text(t, encoding="utf-8")
    print(("patched         " if changed else "already current ") + relpath)

# 1) Performer key-stabilization BUG FIX  -> makes RQ1 a fair comparison
patch("src/eco_dysformer/models/attention.py", [
 ("    def _phi(self, x: torch.Tensor) -> torch.Tensor:",
  "    def _phi(self, x: torch.Tensor, is_query: bool) -> torch.Tensor:"),
 ("        # subtract per-row max over feature dim for stability\n"
  "        stab = torch.amax(proj, dim=-1, keepdim=True).detach()",
  "        if is_query:\n"
  "            stab = torch.amax(proj, dim=-1, keepdim=True).detach()\n"
  "        else:\n"
  "            stab = torch.amax(proj, dim=(-2, -1), keepdim=True).detach()"),
 ("self._phi(q)", "self._phi(q, is_query=True)"),
 ("self._phi(k)", "self._phi(k, is_query=False)"),
])

# 2) 10 outer folds (stats power) + extended crossover sweep
patch("configs/stage1.yaml", [
 ("outer_folds: 5", "outer_folds: 10"),
 ("seq_lengths: [3, 8, 16, 32, 64, 128, 256, 512]",
  "seq_lengths: [3, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]"),
])

# 3) smaller crossover batch for memory headroom at long sequences
patch("src/eco_dysformer/eval/rq1_crossover.py",
      [("batch_size: int = 8", "batch_size: int = 4")])

# 4) honest biomarker face-validity (gaze-feature membership, not keyword list)
patch("src/eco_dysformer/explain/lime_explain.py", [
 ("def _jaccard(a: set, b: set) -> float:",
  "def _is_oculomotor_biomarker(flat_name):\n"
  "    from eco_dysformer.features.gaze import GAZE_FEATURE_NAMES\n"
  "    return flat_name.split('__', 1)[-1] in set(GAZE_FEATURE_NAMES)\n\n\n"
  "def _jaccard(a: set, b: set) -> float:"),
 ("n_bio = sum(any(t in f for t in BIOMARKER_TERMS) for f in top_features)",
  "n_bio = sum(_is_oculomotor_biomarker(f) for f in top_features)"),
])
print("\nAll four patches applied. Now re-run run_stage1, then the report cell.")

already current src/eco_dysformer/models/attention.py
already current configs/stage1.yaml
already current src/eco_dysformer/eval/rq1_crossover.py
already current src/eco_dysformer/explain/lime_explain.py

All four patches applied. Now re-run run_stage1, then the report cell.


In [6]:
import os

print(os.path.exists("/kaggle/input/datasets/pjjoseph3000/etdd70"))
print(os.path.exists("/kaggle/input/datasets/pjjoseph3000/etdd70/data/data"))
print(os.path.exists("/kaggle/input/datasets/pjjoseph3000/etdd70/dyslexia_class_label.csv"))
print(os.path.exists("/kaggle/input/datasets/pjjoseph3000/etdd70/rois/rois"))
print(os.path.exists("/kaggle/input/datasets/pjjoseph3000/etdd70/stimuli/images"))

True
True
True
True
True


In [7]:
import os

os.makedirs("outputs/figures", exist_ok=True)

In [8]:
import shutil
import os

# Create ZIPs from the attached dataset folders
shutil.make_archive(
    "/kaggle/working/rois",
    "zip",
    "/kaggle/input/datasets/pjjoseph3000/etdd70/rois/rois"
)

shutil.make_archive(
    "/kaggle/working/stimuli",
    "zip",
    "/kaggle/input/datasets/pjjoseph3000/etdd70/stimuli/images"
)

print("rois.zip exists:", os.path.exists("/kaggle/working/rois.zip"))
print("stimuli.zip exists:", os.path.exists("/kaggle/working/stimuli.zip"))

rois.zip exists: True
stimuli.zip exists: True


In [9]:
!grep -n "rois_zip" configs/stage1.yaml
!grep -n "stimuli_zip" configs/stage1.yaml

21:  rois_zip: /kaggle/working/rois.zip
22:  stimuli_zip: /kaggle/working/stimuli.zip


In [10]:
!PYTHONPATH=src python -m eco_dysformer.data.inspect_dataset



PATHS
  dataset_root   OK   /kaggle/input/datasets/pjjoseph3000/etdd70
  data_dir       OK   /kaggle/input/datasets/pjjoseph3000/etdd70/data/data
  labels_csv     OK   /kaggle/input/datasets/pjjoseph3000/etdd70/dyslexia_class_label.csv
  rois_zip       OK   /kaggle/working/rois.zip
  stimuli_zip    OK   /kaggle/working/stimuli.zip
  outputs_dir    OK   /kaggle/working/EcoDysformer/outputs
  features_dir   OK   /kaggle/working/EcoDysformer/outputs/features
  results_dir    OK   /kaggle/working/EcoDysformer/outputs/results
  figures_dir    OK   /kaggle/working/EcoDysformer/outputs/figures

STRUCTURE (loader asserts)
  subjects discovered : 70
  passages            : ['syllables', 'meaningful', 'pseudo']
  file types          : ['raw', 'fixations', 'saccades', 'metrics']
  manifest rows       : 840 (expect 70 x 3 x 4 = 840)

LABELS
  class distribution  : {'non-dyslexic': 35, 'dyslexic': 35}
  labelled w/o data   : none
  data w/o label      : none

SAMPLE FILES (first subject/passage, h

In [11]:
!pip install -q stanza

In [12]:
from pathlib import Path

cfg = Path("configs/stage1.yaml")

text = cfg.read_text()

text = text.replace(
    "engine: udpipe                  # udpipe | stanza",
    "engine: stanza                   # udpipe | stanza"
)

cfg.write_text(text)

print("Updated.")

Updated.


In [13]:
!grep "engine:" -n configs/stage1.yaml

78:    engine: stanza


In [14]:
import stanza
stanza.download("cs")

2026-07-07 11:21:55 INFO: Downloaded file to /root/stanza_resources/resources.json
2026-07-07 11:21:55 INFO: Downloading default packages for language: cs (Czech) ...
2026-07-07 11:21:55 INFO: File exists: /root/stanza_resources/cs/default.zip
2026-07-07 11:21:57 INFO: Finished downloading models and saved to /root/stanza_resources


In [15]:
!PYTHONPATH=src python -m eco_dysformer.run_stage1


##############################################################################
# 1/8 dataset inspection
##############################################################################

PATHS
  dataset_root   OK   /kaggle/input/datasets/pjjoseph3000/etdd70
  data_dir       OK   /kaggle/input/datasets/pjjoseph3000/etdd70/data/data
  labels_csv     OK   /kaggle/input/datasets/pjjoseph3000/etdd70/dyslexia_class_label.csv
  rois_zip       OK   /kaggle/working/rois.zip
  stimuli_zip    OK   /kaggle/working/stimuli.zip
  outputs_dir    OK   /kaggle/working/EcoDysformer/outputs
  features_dir   OK   /kaggle/working/EcoDysformer/outputs/features
  results_dir    OK   /kaggle/working/EcoDysformer/outputs/results
  figures_dir    OK   /kaggle/working/EcoDysformer/outputs/figures

STRUCTURE (loader asserts)
  subjects discovered : 70
  passages            : ['syllables', 'meaningful', 'pseudo']
  file types          : ['raw', 'fixations', 'saccades', 'metrics']
  manifest rows       : 840 (expect 

In [16]:
import shutil; shutil.make_archive("/kaggle/working/stage1_outputs","zip","outputs")
print("Zipped -> /kaggle/working/stage1_outputs.zip  (download it AND click 'Save Version')")

Zipped -> /kaggle/working/stage1_outputs.zip  (download it AND click 'Save Version')


In [17]:
%cd /kaggle/working/EcoDysformer

/kaggle/working/EcoDysformer


In [18]:
import json
import numpy as np
import pandas as pd

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

from pathlib import Path

R = Path("outputs/results")

def load(name):
    p = R / name
    return json.loads(p.read_text()) if p.exists() else None

cv = load("cv_results.json")
summ = load("stage1_summary.json")
lime = load("lime_stability.json")

print("Loaded successfully!")
print(cv.keys())

NumPy : 1.26.4
Pandas: 2.2.2
Loaded successfully!
dict_keys(['seed', 'device', 'n_outer', 'n_inner', 'leakage_checked', 'fold_class_balance', 'arms', 'comparisons', 'operational'])


In [19]:
import json, pandas as pd
from pathlib import Path
R = Path("outputs/results")
load = lambda p: json.loads((R/p).read_text()) if (R/p).exists() else None
cv, summ, lime = load("cv_results.json"), load("stage1_summary.json"), load("lime_stability.json")

print("LEAKAGE:", cv.get("leakage_checked"), "| outer folds:", cv.get("n_outer"), "| device:", cv.get("device"))
print("\n== PER-ARM (mean over folds) ==")
rows=[dict(arm=n, acc=round(a["accuracy_mean"],3),
          ci=f'[{a["accuracy_ci"]["lo"]:.2f},{a["accuracy_ci"]["hi"]:.2f}]',
          f1=round(a["f1_mean"],3), auroc=round(a["auroc_mean"],3),
          ece=round(a["ece_mean"],3), brier=round(a["brier_mean"],3),
          leaves=a["selected_num_leaves"]) for n,a in cv["arms"].items()]
print(pd.DataFrame(rows).to_string(index=False))
for n,a in cv["arms"].items():
    if a["accuracy_mean"]>=0.98: print(f"  !! {n} acc={a['accuracy_mean']:.3f} — suspiciously high")

for tag,key in [("RQ1 performer vs quadratic","RQ1_performer_vs_quadratic"),
                ("RQ2 conditioned vs blind","RQ2_conditioned_vs_blind")]:
    c=cv["comparisons"][key]["accuracy"]
    print(f"\n== {tag} ==\n a={c['mean_a']:.3f} b={c['mean_b']:.3f} "
          f"wilcoxon_p={c['wilcoxon']['pvalue']} diffCI=[{c['paired_diff_ci']['lo']:.3f},{c['paired_diff_ci']['hi']:.3f}]")
    print(" per-fold a:", [round(x,3) for x in c['scores_a']], "\n per-fold b:", [round(x,3) for x in c['scores_b']])

print("\n== CROSSOVER seq_len:", summ["stages"].get("crossover",{}).get("crossover_seq_len"))
if (R/"rq1_crossover.csv").exists(): print(pd.read_csv(R/"rq1_crossover.csv").to_string(index=False))
print("\n== OPERATIONAL =="); 
for k,v in cv.get("operational",{}).items():
    print(f" {k:10s} params={v.get('param_count')} peak_mb={v.get('peak_gpu_mem_mb')} lat_ms={round(v.get('latency_ms_mean',0),3)}")
if lime: print("\n== LIME top:", lime["top_features"], "\n jaccard@5:", round(lime["stability_mean_jaccard_topk"],3),
               "spearman:", round(lime["stability_mean_spearman"],3), "biomarker@5:", lime["biomarker_facevalidity_top5"])

LEAKAGE: True | outer folds: 10 | device: cuda

== PER-ARM (mean over folds) ==
                  arm   acc          ci    f1  auroc   ece  brier                         leaves
performer_conditioned 0.829 [0.75,0.90] 0.805  0.861 0.170  0.169 [7, 7, 7, 7, 7, 7, 7, 7, 7, 7]
quadratic_conditioned 0.821 [0.74,0.90] 0.807  0.883 0.178  0.176 [7, 7, 7, 7, 7, 7, 7, 7, 7, 7]
      performer_blind 0.833 [0.74,0.92] 0.807  0.878 0.166  0.164 [7, 7, 7, 7, 7, 7, 7, 7, 7, 7]

== RQ1 performer vs quadratic ==
 a=0.829 b=0.821 wilcoxon_p=0.7864570351373764 diffCI=[-0.088,0.092]
 per-fold a: [0.875, 0.75, 0.875, 0.625, 1.0, 0.667, 0.833, 0.833, 1.0, 0.833] 
 per-fold b: [1.0, 0.75, 0.75, 0.625, 0.75, 1.0, 0.833, 0.833, 1.0, 0.667]

== RQ2 conditioned vs blind ==
 a=0.829 b=0.833 wilcoxon_p=0.8316408424990329 diffCI=[-0.075,0.067]
 per-fold a: [0.875, 0.75, 0.875, 0.625, 1.0, 0.667, 0.833, 0.833, 1.0, 0.833] 
 per-fold b: [0.875, 0.625, 1.0, 0.625, 0.875, 0.833, 0.833, 1.0, 1.0, 0.667]

== CROSSOVER s